# RADKLIM projections

## Summary

* 900km x 900km grid is done.
* 1100km x 900km grid is done.
* 1500x1400 grid is not done because I could not find the upper right coordinates.

In this notebook, I went through the wradlib library source code and used the functions from the wradlib library to get the infos I wanted. Then I build the projection with the obtained proj string and calculated all geographic projection coordinates as well as the projection coordinates. The WKT is extracted directly from wradlib and the CF-conform projection is determined by hand from the PROJ string.

Note that in my opinion, the projection metadata in the RADKLIM data that I found on the opendata Server are not correct.

The data is compatible with the dwd-radolan projection that can be found at:

https://github.com/wradlib/wradlib/blob/main/wradlib/georef/projection.py

This fits the x and y projection coordinates found in the RADKLIM data on the opendata servers to within ca 1 meter. Also the lat and lon fit within a relative tolerance of 0.001

## Reference

The geographic as well as the projection coordinates of the lower left and upper right corners can be found in the source code.

In [ ]:
# https://github.com/wradlib/wradlib/blob/main/wradlib/georef/rect.py

# Coordinates for 1100km x 900km grid
# +------------+-----------+------------+-----------+-----------+
# | Coordinate |   lon     |     lat    |     x     |     y     |
# +============+===========+============+===========+===========+
# | LowerLeft  |  4.6759E  |  46.1929N  | -443.4622 | -4758.645 |
# +------------+-----------+------------+-----------+-----------+
# | LowerRight | 15.4801E  |  46.1827N  |  456.5378 | -4758.645 |
# +------------+-----------+------------+-----------+-----------+
# | UpperRight | 17.1128E  |  55.5342N  |  456.5378 | -3658.645 |
# +------------+-----------+------------+-----------+-----------+
# | UpperLeft  |  3.0889E  |  55.5482N  | -443.4622 | -3658.645 |
# +------------+-----------+------------+-----------+-----------+

# Coordinates for 900km x 900km grid

# +------------+-----------+------------+-----------+-----------+
# | Coordinate |   lon     |     lat    |     x     |     y     |
# +============+===========+============+===========+===========+
# | LowerLeft  |  3.5889E  |  46.9526N  | -523.4622 | -4658.645 |
# +------------+-----------+------------+-----------+-----------+
# | LowerRight | 14.6209E  |  47.0705N  |  376.5378 | -4658.645 |
# +------------+-----------+------------+-----------+-----------+
# | UpperRight | 15.7208E  |  54.7405N  |  376.5378 | -3758.645 |
# +------------+-----------+------------+-----------+-----------+
# | UpperLeft  |  2.0715E  |  54.5877N  | -523.4622 | -3758.645 |
# +------------+-----------+------------+-----------+-----------+

# Coordinates for 1500km x 1400km grid

# +------------+-----------+------------+-----------+-----------+
# | Coordinate |   lon     |     lat    |     x     |     y     |
# +============+===========+============+===========+===========+
# | LowerLeft  |  2.3419E  |  43.9336N  | -673.4622 | -5008.645 |
# +------------+-----------+------------+-----------+-----------+

## All proj string from wradlib

Here I went through the source code and looked for all projections that I could find

In [ ]:
import wradlib.georef.rect as rect
import wradlib.georef.projection as wradlib_projection
import wradlib.georef as georef
import pyku
import xarray as xr

ds = pyku.resources.get_test_data('radolan')
print("opendata                  ", ds.crs.attrs['proj4'])
print("dwd-radolan               ", georef.create_osr("dwd-radolan-sphere").ExportToProj4())
print("dwd-radolan-sphere        ", georef.create_osr("dwd-radolan-sphere").ExportToProj4())
print("dwd-radolan-sphere-rx     ", georef.create_osr("dwd-radolan-sphere-rx").ExportToProj4())
print("dwd-radolan-sphere-de1200 ", georef.create_osr("dwd-radolan-sphere-de1200").ExportToProj4())
print("dwd-radolan-sphere-de4800 ", georef.create_osr("dwd-radolan-sphere-de4800").ExportToProj4())
print("dwd-radolan-wgs84         ", georef.create_osr("dwd-radolan-wgs84").ExportToProj4())
print("dwd-radolan-wgs84-rx      ", georef.create_osr("dwd-radolan-wgs84-rx").ExportToProj4())
print("dwd-radolan-wgs84-de1200  ", georef.create_osr("dwd-radolan-wgs84-de1200").ExportToProj4())
print("dwd-radolan-wgs84-de4800  ", georef.create_osr("dwd-radolan-wgs84-de4800").ExportToProj4())

## CF-conform projection definition

The CF-conform area definition by hand and in yaml format

See: https://github.com/wradlib/wradlib/blob/main/wradlib/georef/projection.py

    polar_stereo_wkt = (
        'PROJECTION["Polar_Stereographic"],'
        'PARAMETER["latitude_of_origin", 60],'
        'PARAMETER["central_meridian", 10],'
        'PARAMETER["false_easting", {0:-.16f}],'
        'PARAMETER["false_northing", {1:-.16f}],'
        "{2}"
        'AXIS["Easting", SOUTH],'
        'AXIS["Northing", SOUTH]]'
    )

CF-conform projection informations

This is done by hand

https://cfconventions.org/Data/cf-conventions/cf-conventions-1.7/build/apf.html

In [ ]:
cf_projection_definition = {
    'grid_mapping_name': 'polar_stereographic',
    'standard_parallel': 60,
    'straight_vertical_longitude_from_pole': 10.0,
    'earth_radius': 6370040.0,
    'false_easting': 0.0,
    'false_northing': 0.0,
    'inverse_flattening': 0.0,
    'long_name': 'RADOLAN Grid',
    'units': 'm'
}

## WKT

The WKT can be taken from wradlib

In [ ]:
wradlib_projection.create_osr('dwd-radolan').ExportToWkt()

## 900x900

### Reference

In [ ]:
wradlib_projection.create_osr('dwd-radolan').ExportToProj4()

In [ ]:
# Coordinates for 900km x 900km grid

ref = """
+------------+-----------+------------+-----------+-----------+
| Coordinate |   lon     |     lat    |     x     |     y     |
+============+===========+============+===========+===========+
| LowerLeft  |  3.5889E  |  46.9526N  | -523.4622 | -4658.645 |
+------------+-----------+------------+-----------+-----------+
| UpperRight | 15.7208E  |  54.7405N  |  376.5378 | -3758.645 |
+------------+-----------+------------+-----------+-----------+
"""

### Calculate

In [ ]:
import pyproj

# Define the geographic coordinates of the lower lef (ll) and upper right (ur) corners
# ------------------------------------------------------------------------------------

lat_ll = 46.9526
lon_ll =  3.5889
lat_ur = 54.7405
lon_ur = 15.7208

# Get projection from wradlib
# ---------------------------

proj_string = wradlib_projection.create_osr('dwd-radolan').ExportToProj4()

# The source projection are the geographic coordinates on a WGS84 earth
# ---------------------------------------------------------------------

wgs84_latlong_proj = pyproj.Proj(proj='latlong', datum='WGS84')

# Create transformer and transform
# ------------------

transformer = pyproj.Transformer.from_proj(proj_from=wgs84_latlong_proj, proj_to=proj_string)

x_ll, y_ll = transformer.transform(lon_ll, lat_ll)
x_ur, y_ur = transformer.transform(lon_ur, lat_ur)

x_ll, y_ll = round(x_ll, 4), round(y_ll, 4)
x_ur, y_ur = round(x_ur, 4), round(y_ur, 4)

# Print for comparison
# --------------------

print("Reference")
print(ref)

print("\nCalculated\n")
print(proj_string)
print("""\
# +------------+-----------+------------+-----------+-----------+
# | Coordinate |   lon     |     lat    |     x     |     y     |
# +============+===========+============+===========+===========+\
"""
)
print(f"# | LowerLeft  | {lon_ll:7.4f}   |  {lat_ll:7.4f}   | {x_ll:-9.4f} | {y_ll:.4f}|")
print(f"# | UpperRight | {lon_ur:7.4f}   |  {lat_ur:7.4f}   | {x_ur:-9.4f} | {y_ur:.4f}|")

lower_left_x, lower_left_y, upper_right_x, upper_right_y = x_ll, y_ll, x_ur, y_ur

### Create area definition for pyku

In [ ]:
from pyresample.geometry import AreaDefinition
area_id = 'DWD_RADOLAN_900x900'
description = 'DWD RADOLAN projection'
proj_id = 'dwd_radolan'
projection = proj_string
width = 900
height = 900
area_extent =  (lower_left_x, lower_left_y, upper_right_x, upper_right_y)
area_def = AreaDefinition(
    area_id,
    description,
    proj_id,
    projection,
    width,
    height,
    area_extent
)

### Area definition for pyku configuration file

In [ ]:
print(area_def.dump())

### Show the map

In [ ]:
area_def.to_cartopy_crs()

## 1100x900

### Reference

In [ ]:
ref = """
# Coordinates for 1100km x 900km grid
# +------------+-----------+------------+-----------+-----------+
# | Coordinate |   lon     |     lat    |     x     |     y     |
# +============+===========+============+===========+===========+
# | LowerLeft  |  4.6759E  |  46.1929N  | -443.4622 | -4758.645 |
# +------------+-----------+------------+-----------+-----------+
# | UpperRight | 17.1128E  |  55.5342N  |  456.5378 | -3658.645 |
# +------------+-----------+------------+-----------+-----------+
"""

### Calculate

In [ ]:
import pyproj

# Define the geographic coordinates of the lower lef (ll) and upper right (ur) corners
# ------------------------------------------------------------------------------------

lat_ll = 46.1929
lon_ll =  4.6759
lat_ur = 55.5342
lon_ur = 17.1128

# Get projection from wradlib
# ---------------------------

proj_string = wradlib_projection.create_osr('dwd-radolan').ExportToProj4()

# The source projection are the geographic coordinates on a WGS84 earth
# ---------------------------------------------------------------------

wgs84_latlong_proj = pyproj.Proj(proj='latlong', datum='WGS84')

# Create transformer and transform
# --------------------------------

transformer = pyproj.Transformer.from_proj(proj_from=wgs84_latlong_proj, proj_to=proj_string)

x_ll, y_ll = transformer.transform(lon_ll, lat_ll)
x_ur, y_ur = transformer.transform(lon_ur, lat_ur)

x_ll, y_ll = round(x_ll, 4), round(y_ll, 4)
x_ur, y_ur = round(x_ur, 4), round(y_ur, 4)

# Print for comparison
# --------------------

print("Reference")
print(ref)

print("\nCalculated\n")
print("""\
# +------------+-----------+------------+-----------+-----------+
# | Coordinate |   lon     |     lat    |     x     |     y     |
# +============+===========+============+===========+===========+\
"""
)
print(f"# | LowerLeft  | {lon_ll:7.4f}   | {lat_ll:7.4f}    | {x_ll:-9.4f} | {y_ll:.4f}|")
print(f"# | UpperRight | {lon_ur:7.4f}   | {lat_ur:7.4f}    | {x_ur:-9.4f} | {y_ur:.4f}|")

lower_left_x, lower_left_y, upper_right_x, upper_right_y = x_ll, y_ll, x_ur, y_ur

### Create area definition for pyku

In [ ]:
from pyresample.geometry import AreaDefinition
area_id = 'DWD_RADOLAN_1100x900'
description = 'DWD RADOLAN projection'
proj_id = 'dwd_radolan'
projection = proj_string
width = 900
height = 1100
area_extent =  (lower_left_x, lower_left_y, upper_right_x, upper_right_y)
area_def = AreaDefinition(
    area_id,
    description,
    proj_id,
    projection,
    width,
    height,
    area_extent
)

### Area definition for pyku configuration file

In [ ]:
print(area_def.dump())

### Show the map

In [ ]:
area_def.to_cartopy_crs()

### Test CF projection definition against data downloaded from opendata

In [ ]:
ds = pyku.resources.get_test_data('radolan')

del ds['crs'].attrs['proj4']
del ds['crs'].attrs['crs_wkt']
del ds['crs'].attrs['spatial_ref']

ds['crs'].attrs = cf_projection_definition
ds['crs'].attrs

#### Latitudes

In [ ]:
import numpy as np

print("From opendata\n")
print(np.flipud(ds.lat.values))

print("\nFrom calculations\n")
print(area_def.get_lonlats()[1])

#### Longitudes

In [ ]:
print("From opendata\n")
print(ds.lon.values)

print("\nFrom calculations\n")
print(area_def.get_lonlats()[0])

#### y

In [ ]:
print("From opendata\n")
print(ds.y.values)

print("\nFrom calculations\n")
calc_x, calc_y = area_def.get_proj_coords()

print(calc_y[:,0][::-1])

#### x

In [ ]:
import numpy as np
np.set_printoptions(threshold=100)

print("From opendata\n")
print(ds.x.values)

print("\nFrom calculations\n")

calc_x, calc_y = area_def.get_proj_coords()

print(calc_x[0,:])

### Test against data downloaded from opendata

Note that the latitude and longitude data are inverted between calculations and the data. imo it is a matter of convention (left to right and bottom to top) which may vary depending on software used. I think

In [ ]:
import xarray as xr
import pyku
import pyku.geo as geo
import pyku.analyse as analyse
import importlib

In [ ]:
ds = pyku.resources.get_test_data('radolan')

#### Projection definition in the data

In [ ]:
ds['crs'].attrs

#### Latitudes

In [ ]:
print("From opendata\n")
print(np.flipud(ds.lat.values))

print("\nFrom calculations\n")
print(area_def.get_lonlats()[1])

#### Longitudes

In [ ]:
print("From opendata\n")
print(ds.lon.values)

print("\nFrom calculations\n")
print(area_def.get_lonlats()[0])

#### y

In [ ]:
print("From opendata\n")
print(ds.y.values)

print("\nFrom calculations\n")
calc_x, calc_y = area_def.get_proj_coords()

print(calc_y[:,0][::-1])

#### x

In [ ]:
import numpy as np
np.set_printoptions(threshold=100)

print("From opendata\n")
print(ds.x.values)

print("\nFrom calculations\n")

calc_x, calc_y = area_def.get_proj_coords()

print(calc_x[0,:])

In [ ]:
ds.crs.attrs

### Reproject in pyku and test alignement against the data

Here I reporject with pyku where the projection is read from the georeferencing and check that the calculations result in the same numbers as the data within tolerance

In [ ]:
reprojected = ds.pyku.project('DWD_RADOLAN_1100x900')

In [ ]:
reprojected.pyku.compare_geographic_alignment(ds, tolerance=0.001)

There is a small difference that is detected

However this is deemed not problematic because a relative tolerance of 0.1 shoule be acceptable

In [ ]:
reprojected.pyku.compare_geographic_alignment(ds, tolerance=0.01)

To make sure there is indeed no issue, I explicitely compare the results

In [ ]:
print("Original data")
print(ds.x.values)

print("Data reprojected with pyku")
print(reprojected.x.values)

In [ ]:
for idx, _ in enumerate(ds.x.values):
    print(f"{ds.x.values[idx]:.4f} - {reprojected.x.values[idx]:.4f} {reprojected.x.values[idx]/ds.x.values[idx]}")